# Phase 0 candidate-constrained division oracle

Parent: `diag_014_train16_preilp_edge_export`. This CPU-only diagnostic uses the fixed 16-video validation set, the official tracking scorer, and the bounded pre-ILP candidate export. It evaluates cumulative families A, A+B, and A+B+C with atomic, conflict-aware edits. Absolute train-derived scores are optimistic; decisions use paired deltas and require positive headroom on both specimens.


In [ ]:
# --- Phase 0.1 : environment ---------------------------------------------------
import importlib, os, sys, glob, subprocess
from pathlib import Path

# Polars 1.x is split into a Python package plus a compiled runtime wheel.
# Prefer the broadly compatible runtime on Kaggle CPUs.
os.environ.setdefault("POLARS_PREFER_PKG", "32")

def _find(pattern):
    return sorted(glob.glob(pattern, recursive=True))

# Support-pack repo = the dir that contains scripts/predict_unet_transformer.py
_hits = _find("/kaggle/input/**/scripts/predict_unet_transformer.py")
assert _hits, "support-pack repo not found under /kaggle/input (attach the pack dataset)."
REPO = Path(_hits[0]).parent.parent
for p in (REPO / "src", REPO / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print("REPO =", REPO)

# Dependency resolution is deliberately disabled so pip cannot replace Kaggle's
# already-imported numpy/scipy and create a binary ABI mismatch. Because --no-deps
# is used, every runtime dependency must be named explicitly. This list is the
# dependency closure validated by the v7 offline gate and the v8 submission.
PIP_SPECS = [
    "tracksdata", "pyscipopt", "ilpy>=0.5.1",
    "zarr>=3.0.10,<4", "geff>=1.1.3.1.1", "geff-spec<1.2",
    "polars>=1.36", "polars-runtime-32>=1.36",
    "blosc2", "dask", "imagecodecs", "scikit-image>=0.24",
    "pyarrow", "rustworkx>=0.17.1", "sqlalchemy>=2", "numcodecs>=0.13,<0.16",
    "donfig>=0.8", "google-crc32c>=1.5", "bidict>=0.23.1", "psygnal>=0.14",
    "rich", "networkx>=3.2.1", "pydantic>=2.11", "pydantic-core",
    "annotated-types", "typing-extensions>=4.13", "typing-inspection",
    "markdown-it-py", "pygments", "click", "cloudpickle", "fsspec",
    "partd", "locket", "toolz", "pyyaml", "ndindex", "msgpack",
    "numexpr", "deprecated", "wrapt",
]

CRITICAL_MODULES = ("tracksdata", "geff", "geff_spec", "zarr",
                    "pyscipopt", "ilpy", "donfig", "numcodecs",
                    "polars", "blosc2", "dask", "imagecodecs",
                    "pyarrow", "rustworkx", "sqlalchemy", "skimage")

def _clear_partial_imports():
    # A failed import can leave half-initialized packages in sys.modules.
    roots = set(CRITICAL_MODULES) | {"skimage"}
    for name in list(sys.modules):
        if any(name == root or name.startswith(root + ".") for root in roots):
            sys.modules.pop(name, None)
    importlib.invalidate_caches()

def _polars_binary_ok():
    # The real test: a py3-none-any polars wheel imports fine but has NO compiled
    # binary ('Polars binary is missing!') -> constructing ANY DataFrame raises
    # `PyDataFrame is not defined`. A Series/DataFrame build catches that; a bare
    # import does not.
    try:
        import polars as _pl
        _pl.DataFrame({"_x": [1]})
        return True
    except Exception:
        return False

def _import_failures():
    failures = {}
    for name in CRITICAL_MODULES:
        try:
            importlib.import_module(name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    try:
        import zarr as _zarr
        if int(_zarr.__version__.split(".")[0]) < 3:
            failures["zarr"] = f"zarr {_zarr.__version__} is too old; need >=3"
    except Exception:
        pass
    try:
        import polars as _pl
        polars_ok = (hasattr(_pl, "Float16") and _polars_binary_ok())
        if not polars_ok:
            failures["polars"] = f"polars {_pl.__version__} unusable (old or binary missing)"
    except Exception:
        pass
    return failures

def _run_pip(command, label):
    print(label)
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
    return result.returncode == 0

failures = _import_failures()
if failures:
    print("Dependency check failed:", failures)
    wheel_dirs = []
    for path in [REPO.parent / "wheels", *map(Path, _find("/kaggle/input/**/wheels"))]:
        if path.is_dir() and path not in wheel_dirs:
            wheel_dirs.append(path)

    base = [sys.executable, "-m", "pip", "install", "--no-deps"]
    installed = False
    if wheel_dirs:
        offline = base + ["--no-index"]
        for path in wheel_dirs:
            offline += ["--find-links", str(path)]
        installed = _run_pip(offline + PIP_SPECS,
                             f"Installing from offline wheels: {wheel_dirs}")
    if not installed:
        installed = _run_pip(base + PIP_SPECS, "Offline install unavailable; trying PyPI")
    if not installed:
        raise RuntimeError("Dependency installation failed; see pip output above.")

    _clear_partial_imports()
    failures = _import_failures()
    non_polars_failures = {k: v for k, v in failures.items() if k != "polars"}
    if non_polars_failures:
        raise ImportError(
            f"Dependencies still fail after installation: {non_polars_failures}"
        )

# polars binary guard: the offline wheels can carry a binary-less polars
# (polars-*-py3-none-any.whl) that SHADOWS Kaggle's working build once installed
# -> "Polars binary is missing!" -> `PyDataFrame is not defined` on every polars
# AND tracksdata DataFrame op. Install the Python package and its compiled runtime
# as a matched pair. Prefer the attached offline wheels, then use PyPI (this audit
# runs with internet ON). No-op when polars already works.
if not _polars_binary_ok():
    print("polars binary missing -> reinstalling polars + polars-runtime-32")
    _clear_partial_imports()
    polars_specs = ["polars>=1.36", "polars-runtime-32>=1.36"]
    repaired = False
    if wheel_dirs:
        command = [sys.executable, "-m", "pip", "install", "--no-deps",
                   "--force-reinstall", "--no-index"]
        for path in wheel_dirs:
            command += ["--find-links", str(path)]
        repaired = _run_pip(command + polars_specs,
                            "Repairing polars from offline wheels")
        _clear_partial_imports()
        repaired = repaired and _polars_binary_ok()
    if not repaired:
        repaired = _run_pip(
            [sys.executable, "-m", "pip", "install", "--no-deps",
             "--force-reinstall", *polars_specs],
            "Repairing polars from PyPI",
        )
    _clear_partial_imports()
    if not repaired or not _polars_binary_ok():
        raise ImportError(
            "polars still has no compiled runtime after reinstall; "
            "check that polars and polars-runtime-32 wheels have matching versions"
        )

import tracksdata, geff, zarr  # noqa: F401
import polars as _pl_check
print("tracksdata", getattr(tracksdata, "__version__", "?"),
      "| geff", getattr(geff, "__version__", "?"),
      "| zarr", getattr(zarr, "__version__", "?"),
      "| polars", _pl_check.__version__, "(binary OK)")

In [ ]:
# Locate the staged oracle module and execute the frozen analysis.
import json
import sys
from pathlib import Path

module_hits = sorted(Path('/kaggle/working').rglob('run_phase0_candidate_oracle.py'))
if not module_hits:
    module_hits = sorted(Path('.').resolve().rglob('run_phase0_candidate_oracle.py'))
assert len(module_hits) == 1, f'Expected one oracle module, found {module_hits}'
sys.path.insert(0, str(module_hits[0].parent))
from run_phase0_candidate_oracle import run

payload = run()
print(json.dumps({
    'gate': payload['metrics']['oracle_gate_passed'],
    'baseline_reproduced': payload['metrics']['baseline_reproduced'],
    'families': payload['metrics']['families'],
}, indent=2, sort_keys=True))


In [ ]:
# Controller output contract: the preceding analysis must write this final artifact.
metrics_path = Path('/kaggle/working/metrics.json')
assert metrics_path.is_file(), 'Oracle did not write metrics.json'
saved_metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
assert saved_metrics['experiment_id'] == 'diag_015_train16_candidate_constrained_oracle'
assert isinstance(saved_metrics['metrics']['oracle_gate_passed'], bool)
print('validated', metrics_path)
